
# Module 1 - Document Ingest with ai_parse_document + ai_query (fixed)

In [ ]:
%run ./_config


In [ ]:
# Pull config from _config.ipynb (already executed via %run); allow per-cell override via widgets.
dbutils.widgets.text("catalog", CATALOG, "Catalog")
dbutils.widgets.text("schema", SCHEMA, "Schema")
dbutils.widgets.text("llm_endpoint", LLM_HAIKU, "LLM endpoint for ai_query")

CATALOG_OVERRIDE = dbutils.widgets.get("catalog")
SCHEMA_OVERRIDE = dbutils.widgets.get("schema")
LLM = dbutils.widgets.get("llm_endpoint")
BASE_LOCAL = f"{CATALOG_OVERRIDE}.{SCHEMA_OVERRIDE}"
VOLUME = f"/Volumes/{CATALOG_OVERRIDE}/{SCHEMA_OVERRIDE}/raw_advisories"
print(f"Using {BASE_LOCAL} with model {LLM}")


## 1. List the raw advisory PDFs

In [ ]:
files = dbutils.fs.ls(VOLUME)
print(f"Found {len(files)} files in {VOLUME}")
for f in files[:10]:
    print(f"  {f.name} ({f.size} bytes)")

## 2. Parse the PDFs with ai_parse_document

In [ ]:
spark.sql(f"""
  CREATE OR REPLACE TABLE {BASE_LOCAL}.parsed_advisories AS
  SELECT path, ai_parse_document(content) AS parsed
  FROM READ_FILES('{VOLUME}', format => 'binaryFile')
""")

spark.sql(f"""
  SELECT path,
         LEFT(CAST(parsed AS STRING), 300) AS parsed_preview
  FROM {BASE_LOCAL}.parsed_advisories
  LIMIT 3
""").display()


## 3. Pull structured threat intel with ai_query

In [ ]:
import json

schema_json = json.dumps({
    "type": "json_schema",
    "json_schema": {
        "name": "advisory_extract",
        "schema": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "summary": {"type": "string"},
                "cves": {"type": "array", "items": {"type": "string"}},
                "vendors": {"type": "array", "items": {"type": "string"}},
                "products": {"type": "array", "items": {"type": "string"}},
                "mitigations": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["title", "summary", "cves"],
        },
        "strict": True,
    },
}).replace("'", "''")

spark.sql(f"""
  CREATE OR REPLACE TABLE {BASE_LOCAL}.structured_advisories AS
  SELECT
    path,
    ai_query(
      '{LLM}',
      CONCAT(
        'Extract structured threat intelligence from this CISA advisory. Return JSON only.\\n\\n',
        LEFT(CAST(parsed AS STRING), 8000)
      ),
      responseFormat => '{schema_json}'
    ) AS extract
  FROM {BASE_LOCAL}.parsed_advisories
""")

spark.sql(f"SELECT COUNT(*) AS rows FROM {BASE_LOCAL}.structured_advisories").display()


## 4. Flatten into queryable columns

In [ ]:
spark.sql(f"""
  CREATE OR REPLACE VIEW {BASE_LOCAL}.advisories AS
  SELECT
    path,
    parse_json(extract):title::string AS title,
    parse_json(extract):summary::string AS summary,
    from_json(parse_json(extract):cves::string, 'ARRAY<STRING>') AS cves,
    from_json(parse_json(extract):vendors::string, 'ARRAY<STRING>') AS vendors,
    from_json(parse_json(extract):products::string, 'ARRAY<STRING>') AS products,
    from_json(parse_json(extract):mitigations::string, 'ARRAY<STRING>') AS mitigations
  FROM {BASE_LOCAL}.structured_advisories
""")

spark.sql(f"SELECT title, cves, vendors FROM {BASE_LOCAL}.advisories LIMIT 10").display()


## 5. Join structured advisories to KEV
FIX: kev_catalog uses column `vendorProject`, not `vendor`.

In [ ]:
spark.sql(f"""
  WITH exploded AS (
    SELECT path, title, EXPLODE(cves) AS cve
    FROM {BASE_LOCAL}.advisories
    WHERE cves IS NOT NULL
  )
  SELECT e.title, e.cve, k.dateAdded, k.dueDate, k.requiredAction
  FROM exploded e
  INNER JOIN {BASE_LOCAL}.kev_catalog k ON e.cve = k.cveID
  ORDER BY k.dateAdded DESC
""").display()
